# 🌍 EarthRanger Data Analysis
## Ranger Patrol Movement Visualization & Analysis

This notebook analyzes trajectory and relocation data exported from the Ecoscope GUI for EarthRanger.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster, AntPath
from shapely import wkt
from shapely.geometry import Point, LineString
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries imported successfully!")

## 2. Load and Inspect Data

In [ ]:
# Load the data files
relocations_df = pd.read_csv('relocations.csv')
trajectories_df = pd.read_csv('trajectories.csv')

print("📍 RELOCATIONS DATA")
print(f"   Shape: {relocations_df.shape}")
print(f"   Columns: {len(relocations_df.columns)}")
print("\n📊 TRAJECTORIES DATA")
print(f"   Shape: {trajectories_df.shape}")
print(f"   Columns: {len(trajectories_df.columns)}")

In [ ]:
# Preview relocations data
print("📍 Relocations - First 5 rows (key columns):")
key_cols_reloc = ['id', 'extra__subject__name', 'fixtime', 'geometry', 
                   'extra__location__latitude', 'extra__location__longitude']
available_cols = [c for c in key_cols_reloc if c in relocations_df.columns]
relocations_df[available_cols].head()

In [ ]:
# Preview trajectories data
print("📊 Trajectories - First 5 rows (key columns):")
key_cols_traj = ['id', 'segment_start', 'segment_end', 'dist_meters', 
                  'speed_kmhr', 'heading', 'geometry', 'extra__subject__name']
available_cols = [c for c in key_cols_traj if c in trajectories_df.columns]
trajectories_df[available_cols].head()

## 3. Data Preprocessing and Cleaning

In [ ]:
# Convert datetime columns (using ISO8601 format for mixed formats)
relocations_df['fixtime'] = pd.to_datetime(relocations_df['fixtime'], format='ISO8601')

trajectories_df['segment_start'] = pd.to_datetime(trajectories_df['segment_start'], format='ISO8601')
trajectories_df['segment_end'] = pd.to_datetime(trajectories_df['segment_end'], format='ISO8601')

# Filter out junk records
reloc_clean = relocations_df[relocations_df['junk_status'] == False].copy()
traj_clean = trajectories_df[trajectories_df['junk_status'] == False].copy()

print(f"📍 Relocations: {len(relocations_df)} → {len(reloc_clean)} (after removing junk)")
print(f"📊 Trajectories: {len(trajectories_df)} → {len(traj_clean)} (after removing junk)")

## 4. Parse Geometry Columns

In [ ]:
# Parse WKT geometry for relocations (POINT)
reloc_clean['geom'] = reloc_clean['geometry'].apply(wkt.loads)

# Parse WKT geometry for trajectories (LINESTRING)
traj_clean['geom'] = traj_clean['geometry'].apply(wkt.loads)

# Extract coordinates for mapping
reloc_clean['lat'] = reloc_clean['extra__location__latitude']
reloc_clean['lon'] = reloc_clean['extra__location__longitude']

print(f"✅ Parsed {len(reloc_clean)} relocation points")
print(f"✅ Parsed {len(traj_clean)} trajectory segments")

# Show sample geometry
print(f"\n📍 Sample relocation point: {reloc_clean['geom'].iloc[0]}")
print(f"📊 Sample trajectory segment: {traj_clean['geom'].iloc[0]}")

## 5. Visualize Relocations on Interactive Map

In [ ]:
# Calculate map center from relocations
center_lat = reloc_clean['lat'].mean()
center_lon = reloc_clean['lon'].mean()

# Create folium map for relocations
reloc_map = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='OpenStreetMap')

# Add marker cluster for relocations
marker_cluster = MarkerCluster().add_to(reloc_map)

# Color map for subjects
subjects = reloc_clean['extra__subject__name'].unique()
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue']
subject_colors = {subj: colors[i % len(colors)] for i, subj in enumerate(subjects)}

# Add markers for each relocation
for idx, row in reloc_clean.iterrows():
    popup_text = f"""
    <b>Subject:</b> {row['extra__subject__name']}<br>
    <b>Time:</b> {row['fixtime']}<br>
    <b>Type:</b> {row['extra__subject__subject_subtype']}<br>
    <b>Lat:</b> {row['lat']:.6f}<br>
    <b>Lon:</b> {row['lon']:.6f}
    """
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=subject_colors.get(row['extra__subject__name'], 'red'),
        fill=True,
        fillColor=subject_colors.get(row['extra__subject__name'], 'red'),
        fillOpacity=0.7,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(marker_cluster)

# Add legend
legend_html = f'''
<div style="position: fixed; bottom: 50px; left: 50px; z-index:9999; 
            background-color:white; padding: 10px; border-radius: 5px; border:2px solid grey;">
<b>📍 Relocations Map</b><br>
'''
for subj, color in subject_colors.items():
    legend_html += f'<i class="fa fa-circle" style="color:{color}"></i> {subj}<br>'
legend_html += f'<br><b>Total Points:</b> {len(reloc_clean)}</div>'

reloc_map.get_root().html.add_child(folium.Element(legend_html))

print(f"📍 Relocation Map - {len(reloc_clean)} points plotted")
print(f"   Location: Vietnam (Ho Chi Minh City area)")
print(f"   Center: {center_lat:.4f}, {center_lon:.4f}")
reloc_map

## 6. Visualize Trajectories on Interactive Map

In [ ]:
# Create trajectory map
traj_map = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='CartoDB positron')

# Sort trajectories by time to show movement order
traj_sorted = traj_clean.sort_values('segment_start')

# Add trajectory lines with color gradient based on speed
from branca.colormap import linear

# Create colormap for speed
speed_colormap = linear.YlOrRd_09.scale(
    traj_clean['speed_kmhr'].min(), 
    traj_clean['speed_kmhr'].max()
)
speed_colormap.caption = 'Speed (km/h)'
speed_colormap.add_to(traj_map)

# Add each trajectory segment
for idx, row in traj_sorted.iterrows():
    line = row['geom']
    coords = list(line.coords)
    # Folium uses [lat, lon] format
    folium_coords = [[coord[1], coord[0]] for coord in coords]
    
    color = speed_colormap(row['speed_kmhr'])
    
    popup_text = f"""
    <b>🚶 Movement Segment</b><br>
    <b>Subject:</b> {row['extra__subject__name']}<br>
    <b>Start:</b> {row['segment_start']}<br>
    <b>End:</b> {row['segment_end']}<br>
    <b>Distance:</b> {row['dist_meters']:.1f} m<br>
    <b>Speed:</b> {row['speed_kmhr']:.2f} km/h<br>
    <b>Heading:</b> {row['heading']:.1f}°<br>
    <b>Duration:</b> {row['timespan_seconds']:.1f} sec
    """
    
    folium.PolyLine(
        folium_coords,
        color=color,
        weight=4,
        opacity=0.8,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(traj_map)

# Add start and end markers
if len(traj_sorted) > 0:
    first_line = traj_sorted.iloc[0]['geom']
    last_line = traj_sorted.iloc[-1]['geom']
    start_coord = list(first_line.coords)[0]
    end_coord = list(last_line.coords)[-1]
    
    folium.Marker(
        [start_coord[1], start_coord[0]],
        popup='🟢 START',
        icon=folium.Icon(color='green', icon='play')
    ).add_to(traj_map)
    
    folium.Marker(
        [end_coord[1], end_coord[0]],
        popup='🔴 END',
        icon=folium.Icon(color='red', icon='stop')
    ).add_to(traj_map)

# Save map to HTML file (to avoid notebook size issues)
traj_map.save('trajectories_map.html')

print(f"📊 Trajectory Map - {len(traj_clean)} segments plotted")
print(f"   Color indicates speed (Yellow=Slow → Red=Fast)")
print(f"\n✅ Map saved to: trajectories_map.html")
print(f"   Open this file in your browser to view the interactive map!")

## 7. Analyze Movement Statistics

In [ ]:
# Calculate summary statistics for movement data
print("=" * 60)
print("📊 MOVEMENT STATISTICS SUMMARY")
print("=" * 60)

# Distance statistics
print("\n📏 DISTANCE TRAVELED (meters)")
print(f"   Total Distance:   {traj_clean['dist_meters'].sum():.2f} m ({traj_clean['dist_meters'].sum()/1000:.3f} km)")
print(f"   Mean per segment: {traj_clean['dist_meters'].mean():.2f} m")
print(f"   Median:           {traj_clean['dist_meters'].median():.2f} m")
print(f"   Min:              {traj_clean['dist_meters'].min():.2f} m")
print(f"   Max:              {traj_clean['dist_meters'].max():.2f} m")
print(f"   Std Dev:          {traj_clean['dist_meters'].std():.2f} m")

# Speed statistics
print("\n🚀 SPEED (km/h)")
print(f"   Mean Speed:       {traj_clean['speed_kmhr'].mean():.2f} km/h")
print(f"   Median Speed:     {traj_clean['speed_kmhr'].median():.2f} km/h")
print(f"   Min Speed:        {traj_clean['speed_kmhr'].min():.2f} km/h")
print(f"   Max Speed:        {traj_clean['speed_kmhr'].max():.2f} km/h")
print(f"   Std Dev:          {traj_clean['speed_kmhr'].std():.2f} km/h")

# Duration statistics
print("\n⏱️ SEGMENT DURATION (seconds)")
print(f"   Total Duration:   {traj_clean['timespan_seconds'].sum():.2f} sec ({traj_clean['timespan_seconds'].sum()/60:.2f} min)")
print(f"   Mean Duration:    {traj_clean['timespan_seconds'].mean():.2f} sec")
print(f"   Median Duration:  {traj_clean['timespan_seconds'].median():.2f} sec")
print(f"   Min Duration:     {traj_clean['timespan_seconds'].min():.2f} sec")
print(f"   Max Duration:     {traj_clean['timespan_seconds'].max():.2f} sec")

print("\n" + "=" * 60)

## 8. Speed Distribution Analysis

In [ ]:
# Create speed distribution visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Speed histogram
ax1 = axes[0, 0]
ax1.hist(traj_clean['speed_kmhr'], bins=20, color='steelblue', edgecolor='white', alpha=0.7)
ax1.axvline(traj_clean['speed_kmhr'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {traj_clean['speed_kmhr'].mean():.2f} km/h")
ax1.axvline(traj_clean['speed_kmhr'].median(), color='orange', linestyle='--', linewidth=2, label=f"Median: {traj_clean['speed_kmhr'].median():.2f} km/h")
ax1.set_xlabel('Speed (km/h)')
ax1.set_ylabel('Frequency')
ax1.set_title('🚀 Speed Distribution')
ax1.legend()

# 2. Speed boxplot
ax2 = axes[0, 1]
bp = ax2.boxplot(traj_clean['speed_kmhr'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
ax2.set_ylabel('Speed (km/h)')
ax2.set_title('📊 Speed Box Plot')

# 3. Distance histogram
ax3 = axes[1, 0]
ax3.hist(traj_clean['dist_meters'], bins=20, color='forestgreen', edgecolor='white', alpha=0.7)
ax3.axvline(traj_clean['dist_meters'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {traj_clean['dist_meters'].mean():.2f} m")
ax3.set_xlabel('Distance (meters)')
ax3.set_ylabel('Frequency')
ax3.set_title('📏 Distance Distribution')
ax3.legend()

# 4. Duration histogram
ax4 = axes[1, 1]
ax4.hist(traj_clean['timespan_seconds'], bins=20, color='coral', edgecolor='white', alpha=0.7)
ax4.axvline(traj_clean['timespan_seconds'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {traj_clean['timespan_seconds'].mean():.2f} sec")
ax4.set_xlabel('Duration (seconds)')
ax4.set_ylabel('Frequency')
ax4.set_title('⏱️ Segment Duration Distribution')
ax4.legend()

plt.suptitle('Movement Metrics Distribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Temporal Analysis of Movement

In [ ]:
# Temporal analysis - Sort by time
traj_sorted = traj_clean.sort_values('segment_start').reset_index(drop=True)

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# 1. Speed over time
ax1 = axes[0]
ax1.plot(traj_sorted['segment_start'], traj_sorted['speed_kmhr'], 
         marker='o', markersize=4, linewidth=1, color='steelblue', alpha=0.7)
ax1.fill_between(traj_sorted['segment_start'], traj_sorted['speed_kmhr'], alpha=0.3)
ax1.set_ylabel('Speed (km/h)')
ax1.set_title('🚀 Speed Over Time')
ax1.grid(True, alpha=0.3)

# 2. Distance per segment over time
ax2 = axes[1]
ax2.bar(traj_sorted['segment_start'], traj_sorted['dist_meters'], 
        width=0.001, color='forestgreen', alpha=0.7)
ax2.set_ylabel('Distance (meters)')
ax2.set_title('📏 Distance per Segment Over Time')
ax2.grid(True, alpha=0.3)

# 3. Cumulative distance over time
ax3 = axes[2]
cumulative_dist = traj_sorted['dist_meters'].cumsum()
ax3.plot(traj_sorted['segment_start'], cumulative_dist, 
         linewidth=2, color='darkred', marker='o', markersize=3)
ax3.fill_between(traj_sorted['segment_start'], cumulative_dist, alpha=0.3, color='coral')
ax3.set_ylabel('Cumulative Distance (m)')
ax3.set_xlabel('Time')
ax3.set_title('📈 Cumulative Distance Over Time')
ax3.grid(True, alpha=0.3)

plt.suptitle('Temporal Movement Analysis', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Print time range
print(f"\n📅 Data Time Range:")
print(f"   Start: {traj_sorted['segment_start'].min()}")
print(f"   End:   {traj_sorted['segment_end'].max()}")
total_time = (traj_sorted['segment_end'].max() - traj_sorted['segment_start'].min())
print(f"   Total Span: {total_time}")

## 10. Subject-wise Movement Summary

In [ ]:
# Group by subject and calculate metrics
subject_summary = traj_clean.groupby('extra__subject__name').agg({
    'dist_meters': ['sum', 'mean', 'count'],
    'speed_kmhr': ['mean', 'max', 'min'],
    'timespan_seconds': ['sum', 'mean']
}).round(2)

# Flatten column names
subject_summary.columns = [
    'Total Distance (m)', 'Avg Distance/Segment (m)', 'Num Segments',
    'Avg Speed (km/h)', 'Max Speed (km/h)', 'Min Speed (km/h)',
    'Total Time (sec)', 'Avg Segment Duration (sec)'
]

# Add km and minutes columns
subject_summary['Total Distance (km)'] = (subject_summary['Total Distance (m)'] / 1000).round(3)
subject_summary['Total Time (min)'] = (subject_summary['Total Time (sec)'] / 60).round(2)

print("👥 SUBJECT-WISE MOVEMENT SUMMARY")
print("=" * 80)
subject_summary

In [ ]:
# Create summary visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Prepare data for plotting
subjects = subject_summary.index.tolist()
colors = plt.cm.Set2(np.linspace(0, 1, len(subjects)))

# 1. Total Distance by Subject
ax1 = axes[0, 0]
bars1 = ax1.bar(subjects, subject_summary['Total Distance (km)'], color=colors)
ax1.set_ylabel('Distance (km)')
ax1.set_title('📏 Total Distance Traveled by Subject')
ax1.tick_params(axis='x', rotation=45)
for bar, val in zip(bars1, subject_summary['Total Distance (km)']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.2f}', ha='center', va='bottom', fontsize=10)

# 2. Average Speed by Subject
ax2 = axes[0, 1]
bars2 = ax2.bar(subjects, subject_summary['Avg Speed (km/h)'], color=colors)
ax2.set_ylabel('Speed (km/h)')
ax2.set_title('🚀 Average Speed by Subject')
ax2.tick_params(axis='x', rotation=45)
for bar, val in zip(bars2, subject_summary['Avg Speed (km/h)']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1f}', ha='center', va='bottom', fontsize=10)

# 3. Number of Segments by Subject
ax3 = axes[1, 0]
bars3 = ax3.bar(subjects, subject_summary['Num Segments'], color=colors)
ax3.set_ylabel('Number of Segments')
ax3.set_title('📊 Number of Movement Segments by Subject')
ax3.tick_params(axis='x', rotation=45)
for bar, val in zip(bars3, subject_summary['Num Segments']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{int(val)}', ha='center', va='bottom', fontsize=10)

# 4. Total Time by Subject
ax4 = axes[1, 1]
bars4 = ax4.bar(subjects, subject_summary['Total Time (min)'], color=colors)
ax4.set_ylabel('Time (minutes)')
ax4.set_title('⏱️ Total Tracking Time by Subject')
ax4.tick_params(axis='x', rotation=45)
for bar, val in zip(bars4, subject_summary['Total Time (min)']):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{val:.1f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Subject-wise Movement Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 📋 Analysis Summary & Key Findings

In [ ]:
# Generate final summary report
print("=" * 70)
print("🌍 EARTHRANGER DATA ANALYSIS - FINAL SUMMARY")
print("=" * 70)

print("\n📍 DATA OVERVIEW")
print(f"   • Total Relocation Points: {len(reloc_clean)}")
print(f"   • Total Trajectory Segments: {len(traj_clean)}")
print(f"   • Unique Subjects: {traj_clean['extra__subject__name'].nunique()}")
print(f"   • Subject Type: {traj_clean['extra__subject__subject_subtype'].iloc[0]}")

print("\n🗺️ GEOGRAPHIC COVERAGE")
print(f"   • Location: Ho Chi Minh City area, Vietnam")
print(f"   • Latitude Range: {reloc_clean['lat'].min():.4f} - {reloc_clean['lat'].max():.4f}")
print(f"   • Longitude Range: {reloc_clean['lon'].min():.4f} - {reloc_clean['lon'].max():.4f}")

print("\n📊 KEY MOVEMENT METRICS")
print(f"   • Total Distance Covered: {traj_clean['dist_meters'].sum()/1000:.3f} km")
print(f"   • Average Speed: {traj_clean['speed_kmhr'].mean():.2f} km/h")
print(f"   • Maximum Speed: {traj_clean['speed_kmhr'].max():.2f} km/h")
print(f"   • Total Tracking Duration: {traj_clean['timespan_seconds'].sum()/60:.2f} minutes")

print("\n📅 TEMPORAL COVERAGE")
print(f"   • First Record: {traj_clean['segment_start'].min()}")
print(f"   • Last Record: {traj_clean['segment_end'].max()}")

print("\n💡 KEY OBSERVATIONS")
print("   • Data represents ranger team patrol movements")
print("   • Movement patterns show typical patrol activity")
print("   • Speed variations indicate walking/vehicle movement")
print("   • Geographic spread covers urban patrol area")

print("\n" + "=" * 70)
print("✅ Analysis Complete!")
print("=" * 70)